# Soccer

"We lost because we didn't win." -- Cristiano Ronaldo

"Some people think football is a matter of life and death. I assure you, it's much more serious than that." -- Bill Shankly

"The vision of a champion is bent over, drenched in sweat, at the point of exhaustion, when nobody else is looking." -- Anson Dorrance

## Python Modules

We will use the following Python modules in this chapter that we have used in
the past:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import isapy_data

We will also introduce the following new modules from `scikit-learn` and
`scipy` for clustering analysis:

In [ ]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.linear_model import LinearRegression
from scipy.stats import t as t_dist

In [ ]:
#| echo: false

sns.set_theme(style="whitegrid")

## If you don't know anything about soccer

Soccer[^Soccer-1] is the most popular team sport in the world. It flourishes
in both the wealthiest of nations, such as Qatar, host of the 2022 World Cup,
and the poorest of nations, where children often play without shoes or grass,
and perhaps a ball made of recycled or discarded materials. At its most basic
level, soccer is played between two teams each consisting of 11 people on a
field approximately 100 meters long by 50 meters wide, with a goal at each end
that is 2.44 meters high and 7.32 meters wide (or 8 feet by 24 feet). The
object is to advance a ball down the field and into the goal using any part of
the body except the hands and arms. One of the 11 players on the team is
designated as the goalkeeper, or goalie. The goalkeeper wears a different
colored shirt to designate his position, as he is the one player on the team
who is allowed to use his hands and arms to handle the ball.

[^Soccer-1]: While soccer is called "football" in much of the world, we will
refer to it as soccer so as not to confuse it with American Football in a
previous chapter.

In [ ]:
#| fig-cap: "Diagram of a soccer field (pitch)."
#| label: fig-soccer-pitch
#| echo: false

from IPython.display import Image, display
display(Image(filename="fig/soccer-pitch.jpeg", width=600))

For a more in-depth explanation of the laws of soccer, we refer you to
[https://www.soccercoachingpro.com/soccer-rules/](https://www.soccercoachingpro.com/soccer-rules/)
and if you would like to read the laws themselves as kept by the governing body
FIFA, you can find them at
[https://www.theifab.com/laws-of-the-game-documents/?language=all&year=2022%2F23](https://www.theifab.com/laws-of-the-game-documents/?language=all&year=2022%2F23).

Perhaps the simplicity of the game is why it is so popular. There are but 17
basic rules (laws) to the game, and those laws govern the play of soccer
worldwide. While you might think that with a game this simple that there is
little place for analytics, its popularity and the money involved means that
any possible advantage will be sought, and this includes through analytics.

## Soccer Data

In this chapter, we will explore soccer data using methods we have used
before, but we will also learn several unsupervised learning methods, *e.g.*
hierarchical clustering, to apply to soccer analytics.

### FB Reference {.unnumbered}

In October 2022, the Sports Reference website announced a partnership with
Stats Perform Opta to dramatically increase the quantity and quality of soccer
data that are available on its site. Soccer data are now hosted at
[https://fbref.com/en/](https://fbref.com/en/) and includes data on most major
professional leagues throughout the world at a team and player level. More
details related to this new partnership can be found at
[https://www.sports-reference.com/blog/2022/10/fbref-leagues-🇵🇹-leagues-🇧🇷-leagues-🇲🇽-expanded-womens-and-mens-data-new-data-partner/](https://www.sports-reference.com/blog/2022/10/fbref-leagues-🇵🇹-leagues-🇧🇷-leagues-🇲🇽-expanded-womens-and-mens-data-new-data-partner/).

Unfortunately, FBRef and Opta were unable to come to terms on a data-sharing
renewal, and as a result FBRef no longer provides advanced statistics such as
xG and xA on its website. The NWSL player statistics used in this chapter were
collected prior to this change and are available through the book's `isapy_data`
module.

In this chapter, we will focus on using soccer data to cluster both teams and
individual players.

### Expected Goals and Assists {.unnumbered}

Soccer is notoriously low-scoring. In the 2022 World Cup in Qatar, there were
172 goals scored over the 64 matches, for a mean of 2.68 goals per match. This
was the highest scoring World Cup ever, and even so there were seven matches
that were scoreless. Moreover, there is certainly an element of luck in scoring
a goal, and sometimes there are own goals, where a player inadvertently puts
the ball into his own net. Therefore, one of the main developments in the
soccer-analytics space is the introduction of Expected Goals, or xG for short,
as a means of measuring likelihood of success on a given shot.

There are various models to compute xG, but at a high level xG is simply the
probability of scoring a goal from a given spot on the field when a shot is
taken. Other factors that can be taken into account is whether the shot came
from the head or the foot, how the shooter received the ball (cross, set piece,
etc.), position of the goalkeeper, position of other players (teammates and
opponents) between the shooter and the goal, how tightly the defenders are
playing (marking) the shooter, and a host of other situational variables that
can be measured and calculated.

Given that most shots do not result in a goal, most passes to a shot taker will
likely lead to a missed goal as well. Therefore, there is an analogous result
for assists that is referred to as Expected Assists, or xA for short. You will
often see these metrics normalized to ninety minutes so that a player who is a
substitute and only plays thirty minutes can be directly compared to a starter
who might play sixty to ninety minutes. The normalized expected goals and
assists are often written as xG (xA) per ninety (minutes).

A player who is consistently achieving a high number of xG will be one who is
getting himself into a good position consistently on the field. Coaches and
scouts can use this to evaluate whether a player is exceedingly (un)lucky over
a given number of games and will help in evaluating that player's offensive
skills beyond simple counts.

### Player Statistics {.unnumbered}

We will initially focus our attention on individual players from the US
National Women's Soccer League (NWSL). The NWSL is the largest women's soccer
league in North America. In the 2024 season there were fourteen teams and
the Orlando Pride were crowned champion.

You might wonder why we examine a relatively minor league like the NWSL as
compared with the English Premier League (EPL), German Bundesliga, Spanish
La Liga, Italian Serie A, or even Major League Soccer in the US. There is
certainly more global interest in those leagues and likely a higher volume and
quality of data for those leagues. However, in data science the greatest
insights can often be gained by looking where others are not. Moreover the
talent in the NWSL is more evenly spread than in leagues like the EPL where
there are the "big five" clubs and everyone else, or the Bundesliga where in
the 25 seasons from 1999 to 2023 Bayern Munich won the title 19 times,
including the last 11 consecutive seasons, and finished second on three of the
other six occasions. The evenness of talent lends itself then to investigation
by cluster analysis, such that we can identify which players on which teams are
most similar, and can be used for talent acquisition by future team
administrators.

For this textbook, we obtained a .csv file of 2024 NWSL player statistics
from fbref.com and those data are supplied in the book's `isapy_data` module
as `nwsl_player_stats_2024`. For reference, the code for transforming from the
original `.csv` to the format in the module is given below.

In [ ]:
#| eval: false

nwsl_player_stats = (
    pd.read_csv("nwsl-players.csv")
    .rename(columns={
        "xG_1": "xGp90", "xAG_1": "xAp90", "xG_xAG": "xGxAp90",
        "npxG_1": "npxGp90", "npxG_xAG_1": "npxGxAp90"
    })
    [["player", "pos", "squad", "age", "min", "starts",
      "xGp90", "xAp90", "xGxAp90", "npxGp90", "npxGxAp90"]]
)

In [ ]:
nwsl_player_stats = isapy_data.load_data("nwsl_player_stats_2024")
nwsl_player_stats.head()

In [ ]:
nwsl_player_stats.shape

### Exploratory Data Analysis {.unnumbered}

We will begin by exploring expected goals. In particular, let's create a plot
of xG vs age in the NWSL. We will use the `xGp90` column as our xG value to
normalize the number of minutes played across all players. In addition, we
filter out goalkeepers as they are not likely to score or assist on a goal.

In [ ]:
#| label: fig-soccer-xgp90
#| fig-cap: "Player age (x) versus expected goals per 90 minutes played (y) during the 2024 NWSL season."

nwsl_no_gk = nwsl_player_stats.query("pos != 'GK'").copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(nwsl_no_gk["age"], nwsl_no_gk["xGp90"], alpha=0.5)
ax.set_xlabel("Age of Player")
ax.set_ylabel("Expected Goals per 90 (xGp90)")
ax.set_title("xG per 90 by Player Age")
plt.show()

We can see many of the players are clustered between 21 and 31 years old and
between 0 and 0.3 `xGp90`. However, there are certainly outlier players who
are young or with relatively high `xGp90`. We can identify the players by name
on the plot by adding labels to certain points. Suppose we want to identify
players who are less than 20 years of age with a positive `xG` value as well
as all players with an `xG` value greater than 0.6.

In [ ]:
#| label: fig-soccer-xgp90-labels
#| fig-cap: "Player age (x) versus expected goals per 90 minutes played (y) during the 2024 NWSL season with labels for players less than 20 years of age or an expected goals per 90 above 0.6."

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(nwsl_no_gk["age"], nwsl_no_gk["xGp90"], alpha=0.5)

label_mask = ((nwsl_no_gk["age"] < 20) & (nwsl_no_gk["xGp90"] > 0)) | \
             (nwsl_no_gk["xGp90"] > 0.6)
labeled = nwsl_no_gk[label_mask]

for _, row in labeled.iterrows():
    ax.annotate(row["player"], (row["age"], row["xGp90"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.set_xlabel("Age of Player")
ax.set_ylabel("Expected Goals per 90 (xGp90)")
ax.set_title("xG per 90 by Player Age")
plt.show()

These players identified on the plot are either currently offensive stars or
perhaps will be soon!

::: {layout-ncol=2}
![Smith](fig/smith.jpeg)

![Rodman](fig/rodman.jpeg)
:::

## Clustering

After we plotted `xGp90` for individual players in the above section, we may
now want to classify different groups of players. In the topic of soccer
analytics, we will focus on several analytical tools that fall under the
statistical, or machine, learning umbrella of unsupervised learning.
Unsupervised learning is generally thought to be exploratory data analytic
techniques that are designed to learn as much as possible about the data
without the aid of labels as is the case in a regression or classification
model (supervised learning). Clustering, in particular, refers to the class of
techniques concerned with finding groups within your data set that are similar
(in some sense) with respect to the measured variables (or covariates).

### K-Means Clustering {.unnumbered}

In short, K-means clustering of data refers to assigning each observation in
the data set to one of K clusters. The number of clusters, K is specified *a
priori* by the analyst. The algorithm for computing a K-means cluster of n
observations is relatively simple and intuitive, and easy to visualize when
using two or three dimensions. Each observation is randomly assigned to one of
the clusters and the "center" of each cluster is computed, *e.g.* the mean
vector in K-means. Next, you assign each observation to the closest "center",
recompute the "center", and reassign each observation until observations stop
being reassigned. For more in-depth explanations of K-means clustering, you
can visit the section on K-Means clustering at the companion for the book
Introduction to Statistical Learning Using R by Gareth James, Daniela Witten,
Trevor Hastie, and Rob Tibshirani book at
[https://r4ds.github.io/bookclub-islr/k-means.html](https://r4ds.github.io/bookclub-islr/k-means.html).

An important preprocessing step before applying any distance-based clustering
algorithm is to **standardize** the features. Variables measured on different
scales (*e.g.*, age ranging from 18--35, minutes played from 0--3000, and
xGp90 from 0--1) will otherwise allow large-magnitude features to dominate the
distance calculations. We use `StandardScaler` from `scikit-learn` to
transform each feature to have zero mean and unit variance before clustering.

### Clustering Expected Goals and Assists with Age {.unnumbered}

First let's look at a clustering based on age of the player (`age`), total
minutes played in the season (`min`), and expected goals and assists per ninety
minutes played (`xGp90` and `xAp90`, respectively). We are going to consider
three clusters initially, filter out goalkeepers, and also only include players
with `xGxAp90` of less than 1.5 to remove some outliers with little playing
time that may have had good scoring chances in those few minutes and distort
the overall results.

In [ ]:
nwsl_cluster_df = (
    nwsl_player_stats[["player", "squad", "pos", "age", "min", "starts",
                        "xGp90", "xAp90", "xGxAp90", "npxGp90", "npxGxAp90"]]
    .query("xGxAp90 < 1.5 and pos != 'GK'")
    .dropna()
    .copy()
)

features_1 = nwsl_cluster_df[["age", "min", "xGp90", "xAp90"]]
scaler_1 = StandardScaler()
X_scaled_1 = scaler_1.fit_transform(features_1)

km_1 = KMeans(n_clusters=3, random_state=9019, n_init=10)
nwsl_cluster_df["k_means"] = km_1.fit_predict(X_scaled_1)

How can we visualize these clusters? We are not able to visualize all the
dimensions at once given the two-dimensional nature of this book. Understanding
those limitations of visualization, let's look at some plots. First, we plot
age versus minutes played with differing colors based on their cluster
membership in @fig-soccer-age-min.

In [ ]:
#| label: fig-soccer-age-min
#| fig-cap: "Player age (x) versus minutes played (y) during the 2024 NWSL season. Colors represent cluster membership using a K-means clustering with three clusters."

fig, ax = plt.subplots(figsize=(8, 5))
markers = ["o", "s", "^"]
for i, cluster in enumerate(sorted(nwsl_cluster_df["k_means"].unique())):
    subset = nwsl_cluster_df[nwsl_cluster_df["k_means"] == cluster]
    ax.scatter(subset["age"], subset["min"], marker=markers[i],
               label=f"Cluster {cluster + 1}", alpha=0.7)
ax.set_xlabel("Age of Player")
ax.set_ylabel("Minutes Played")
ax.set_title("NWSL players clustered by PT and Age")
ax.legend(title="Cluster")
plt.show()

What stands out amongst this clustering of points? Even after standardizing the
features, the clusters are still largely influenced by minutes played. This is
because the variable `min` has a natural grouping structure (starters vs.
substitutes vs. limited minutes) that the algorithm picks up.

Is anything else evident if we omit minutes played and plot age of the player
versus expected goals (or assists)? @fig-soccer-age-xg-cl shows the cluster
memberships for `age` versus `xGp90`.

In [ ]:
#| label: fig-soccer-age-xg-cl
#| fig-cap: "Player age (x) versus expected goals per 90 minutes played (y) during the 2024 NWSL season. Colors represent cluster membership using a K-means clustering with three clusters."

fig, ax = plt.subplots(figsize=(8, 5))
for i, cluster in enumerate(sorted(nwsl_cluster_df["k_means"].unique())):
    subset = nwsl_cluster_df[nwsl_cluster_df["k_means"] == cluster]
    ax.scatter(subset["age"], subset["xGp90"], marker=markers[i],
               label=f"Cluster {cluster + 1}", alpha=0.7)
ax.set_xlabel("Age of Player")
ax.set_ylabel("Expected Goals per 90")
ax.set_title("NWSL players clustered by xGp90 and Age")
ax.legend(title="Cluster")
plt.show()

Nothing is clearly evident from this figure, so we should do further analysis.
Given that minutes played dominates the previous clustering, consider a K-means
clustering on `age`, expected goals, and expected assists, both normalized per
ninety minutes played and construct a similar plot as above.

In [ ]:
features_2 = nwsl_cluster_df[["age", "xGp90", "xAp90"]]
scaler_2 = StandardScaler()
X_scaled_2 = scaler_2.fit_transform(features_2)

km_2 = KMeans(n_clusters=3, random_state=9019, n_init=10)
nwsl_cluster_df["clusters_two"] = km_2.fit_predict(X_scaled_2)

In [ ]:
#| label: fig-soccer-age-xg-cl-2
#| fig-cap: "Player age (x) versus expected goals per 90 minutes played (y) during the 2024 NWSL season. Colors represent cluster membership using a K-means clustering with three clusters (no minutes played)."

fig, ax = plt.subplots(figsize=(8, 5))
for i, cluster in enumerate(sorted(nwsl_cluster_df["clusters_two"].unique())):
    subset = nwsl_cluster_df[nwsl_cluster_df["clusters_two"] == cluster]
    ax.scatter(subset["age"], subset["xGp90"], marker=markers[i],
               label=f"Cluster {cluster + 1}", alpha=0.7)
ax.set_xlabel("Age of Player")
ax.set_ylabel("Expected Goals per 90")
ax.set_title("NWSL players clustered by xGp90 and Age")
ax.legend(title="Cluster")
plt.show()

As you might imagine, the clustering is now influenced by the age variable.
Even though we have standardized the features, `age` still has a strong
natural grouping structure that the algorithm identifies.

It's apparent that including variables like `age` and `minutes` tends to
dominate the makeup of the clusters. So, if we want to identify similar
players based on their scoring ability, let's only consider the appropriate
metrics. Here we perform an additional clustering using expected goals and
assists (appropriately normalized) and plot the results.

In [ ]:
features_3 = nwsl_cluster_df[["xGp90", "xAp90"]]
scaler_3 = StandardScaler()
X_scaled_3 = scaler_3.fit_transform(features_3)

km_3 = KMeans(n_clusters=3, random_state=9019, n_init=10)
nwsl_cluster_df["clusters_three"] = km_3.fit_predict(X_scaled_3)

In [ ]:
#| label: fig-soccer-xg-xa-cl-3
#| fig-cap: "Expected assists per 90 (x) versus expected goals per 90 minutes played (y) during the 2024 NWSL season. Colors represent cluster membership using a K-means clustering with three clusters."

fig, ax = plt.subplots(figsize=(8, 5))
for i, cluster in enumerate(sorted(nwsl_cluster_df["clusters_three"].unique())):
    subset = nwsl_cluster_df[nwsl_cluster_df["clusters_three"] == cluster]
    ax.scatter(subset["xAp90"], subset["xGp90"], marker=markers[i],
               label=f"Cluster {cluster + 1}", alpha=0.7)
ax.set_xlabel("Expected Assists per 90")
ax.set_ylabel("Expected Goals per 90")
ax.set_title("NWSL players clustered by xGp90 and xAp90")
ax.legend(title="Cluster")
plt.show()

What if we wanted to further examine by position? We can change the color
and/or shape of the points based on position of the players. In this case for
simplicity, we change the original position variable in the data set to just
three positions: defender (D), midfielder (M), and forward (F).

In [ ]:
#| label: fig-soccer-xg-xa-cl-4
#| fig-cap: "Expected assists per 90 (x) versus expected goals per 90 minutes played (y) during the 2024 NWSL season. Colors represent cluster membership using a K-means clustering with three clusters and point characters are position."

nwsl_cluster_df["short_pos"] = nwsl_cluster_df["pos"].str[0]
pos_markers = {"D": "o", "M": "s", "F": "^"}

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette("Set1", 3)
for i, cluster in enumerate(sorted(nwsl_cluster_df["clusters_three"].unique())):
    for pos, marker in pos_markers.items():
        subset = nwsl_cluster_df[
            (nwsl_cluster_df["clusters_three"] == cluster) &
            (nwsl_cluster_df["short_pos"] == pos)
        ]
        ax.scatter(subset["xAp90"], subset["xGp90"], marker=marker,
                   color=palette[i], alpha=0.75,
                   label=f"Cluster {cluster + 1}, {pos}")

handles, labels = ax.get_legend_handles_labels()
cluster_handles = [plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=palette[i], markersize=8,
                   label=f"Cluster {i + 1}") for i in range(3)]
pos_handles = [plt.Line2D([0], [0], marker=m, color="w",
               markerfacecolor="gray", markersize=8,
               label=p) for p, m in pos_markers.items()]
ax.legend(handles=cluster_handles + pos_handles,
          title="Cluster / Position", loc="upper right")
ax.set_xlabel("Expected Assists per 90")
ax.set_ylabel("Expected Goals per 90")
ax.set_title("NWSL players clustered by xGp90 and xAp90")
plt.show()

Do certain positions dominate certain clusters? Yes, and this should be
expected. The highest scoring positions are the Forwards, followed by the
Midfielders and finally by the Defense. Similarly, they seem to each be
largely in their own clusters.

___

## Research Block: When and How to Pass the Ball?

As mentioned at the beginning of the chapter, soccer is a notoriously
low-scoring game. With so many games being decided by scores like 1-0 or 2-1,
or ending in a 0-0 or 1-1 draw, every goal matters. It is so low scoring
because it is extremely difficult to move the ball up the field without using
your hands or arms through a defense of ten highly skilled players all trying
to stop the ball from getting further up the field, and a goalkeeper beyond
the defense who is allowed to use his hands and arms to stop the ball as well.
For a team to score, they need to possess the ball, which means keeping it away
from the defending team. When a player becomes guarded (marked) too closely,
usually he will have to kick the ball, either towards the goal as a shot or
towards a teammate as a pass, to avoid being dispossessed of the ball by the
defender. More often than not, he will attempt to pass.

Not all passes are created equally. Sometimes passes are quite aggressive and
move the ball towards the goal, while other times they may be passive, even
passing to a teammate much further away from the goal. The purpose of the pass
may be to try to set up a scoring chance, or it may be just to keep the ball
away from the defense. Some passes also have a higher chance of being completed
than others. Often the passing strategy used by teams is consistent with their
overall strategy, such as one where a team tries to maximize the number of
completed passes constantly looking for a striking opportunity, as opposed to
the counterattacking strategy used by 2016 EPL champions Leicester City where
their success relied more upon the long pass up the field from their defensive
end.

Passing is a difficult metric to measure. One can easily measure the number of
passes attempted and completed by a team, but how is that measure meaningful
when trying to understand how a team can ultimately score more goals than their
opponents. It would be relatively easy for two teammates to stand two meters
apart and pass the ball back and forth to each other, and would likely remain
unmolested by the defense if they were nowhere near the attacking positions.
Indeed, even the title of the paper "Decomposing the Immeasurable Sport: A
deep learning expected possession value framework for soccer", available at
[http://www.lukebornn.com/papers/fernandez_sloan_2019.pdf](http://www.lukebornn.com/papers/fernandez_sloan_2019.pdf),
authors Javier Fernandez, Luke Bornn, and Dan Cervone mention it as such while
they discuss their strategy for measuring pass efficiency.

With player tracking data becoming more available and relevant now, there are
data now ready for analysis and processing of on-field movements and ball
movements. The authors of the paper analyze the data to determine all the
possible passing vectors available to a player possessing the ball at any given
time, and then attempt to calculate the "best" pass that the player could make,
depending on the strategy of the team at that given time. They do this by
defining a new metric, Expected Possession Value (EPV) which ranges from 1 if
the possessing team scores a goal and -1 if the ball is taken from them and the
opposing team immediately scores a goal. Their deep learning model shows how
EPV would increase or decrease based on a player attempting the pass. This
allows for measurement of passes that have high probability of being completed
but perhaps little increase in the chance of scoring a goal vs ones that may
have a lower probability of being completed but yet more greatly increase the
chances of scoring a goal. Indeed the authors compare the 2010 and 2018
Spanish national teams, which employed the same "tiki-taka" strategy of moving
the ball around but by 2018 were merely completing a lot of low-value passes.

The EPV framework provides a look into a possible expansion over xG and xA
that we have discussed earlier in this chapter and breaks it down into
opportunities for moving the ball into those higher value positions.

___

## Hierarchical Clustering

Hierarchical clustering methods are a commonly used class of alternatives to
k-means clustering. As the name suggests, the methodology will produce a
"hierarchy" amongst the observations relative to some distance measure, and
the results can be easy to visualize. There are several competing hierarchical
methods from which to choose, however, we will only focus on those that are
agglomerative in nature. That is, each observation is considered its own
cluster initially and are paired up in subsequent steps based on dissimilarity
measures between observations/clusters.

We again standardize the features before computing distances, using the same
`StandardScaler` approach.

In [ ]:
nwsl_hclust_df = nwsl_cluster_df[["player", "squad", "age", "starts",
                                   "xGp90", "xAp90", "xGxAp90",
                                   "npxGp90", "npxGxAp90"]].copy()

features_h = nwsl_hclust_df[["age", "starts", "xGp90", "xAp90",
                              "xGxAp90", "npxGp90", "npxGxAp90"]]
scaler_h = StandardScaler()
X_scaled_h = scaler_h.fit_transform(features_h)

Z = linkage(X_scaled_h, method="average")

@fig-soccer-dendo shows that with a hierarchical clustering, we can ultimately
assign each observation to its own cluster. Obviously, this is not useful and
defeats the purpose of clustering. We have to cut the clustering at a certain
level in order to make this a meaningful methodology.

In [ ]:
#| label: fig-soccer-dendo
#| fig-cap: "Dendrogram based on the hierarchical clustering using average linkage."
#| fig-width: 7

fig, ax = plt.subplots(figsize=(7, 5))
dendrogram(Z, no_labels=True, ax=ax)
ax.set_ylabel("Distance")
ax.set_title("Hierarchical Clustering Dendrogram")
plt.show()

As you can see, it is extremely difficult to visualize the dendrogram in its
entirety. We can, however, visualize specific pieces and try to identify
individuals within clusters. We can easily see how many elements fall into a
particular clustering, *e.g.* five clusters or groups using the `fcluster`
function.

In [ ]:
clusters = fcluster(Z, t=5, criterion="maxclust")
pd.Series(clusters).value_counts().sort_index()

Do we see any patterns in the data with respect to clusters? We can append the
cluster memberships to the data frame used in the clustering. This allows us
to visualize cluster membership in two dimensions as the following plot
illustrates.

In [ ]:
#| label: fig-soccer-hier
#| fig-cap: "Player age (x) versus number of starts (y) with colors representing cluster membership based on the hierarchical clustering using average linkage."

nwsl_hclust_df["clusters"] = clusters

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette("Set1", 5)
for i, cluster in enumerate(sorted(nwsl_hclust_df["clusters"].unique())):
    subset = nwsl_hclust_df[nwsl_hclust_df["clusters"] == cluster]
    ax.scatter(subset["age"], subset["starts"], color=palette[i],
               alpha=0.75, label=f"Cluster {cluster}")
ax.set_xlabel("Age of Players")
ax.set_ylabel("Number of Starts")
ax.set_title("NWSL players clustered by starting frequency")
ax.legend(title="Cluster")
plt.show()

After making this two-dimensional plot, it seems that there is strong
separation between the clusters when we consider player age and the number of
starts. Therefore we can deduce that the hierarchical cluster is dominated by
the `age` and `starts` variables.

## Gaussian Mixtures

This section is one of the more advanced topics in the textbook, however, we
feel that its utility in a host of differing domains is worth noting. The class
of algorithm that we will explore here is often referred to as Gaussian, or
normal, mixture models.

The two preceding clustering algorithms result in a "hard" clustering of
observations into groups as there is no room for ambiguity in the cluster
assignments. We now turn our attention to a probability-based clustering
algorithm using mixtures of K Gaussian distributions. The result is a "soft"
clustering in which each observation is assigned to a cluster using a
probability. We write the probability of observation $\boldsymbol{x}_i$
belonging to cluster $k=1,2,\dots,K$ as
$P(i^{th} \ \textrm{observation} \in k | \boldsymbol{x}_i )$, with

$$
\sum_{k = 1}^K P(i^{th} \ \textrm{observation} \in k | \boldsymbol{x}_i ) = 1.
$$

The Gaussian (Normal) mixture model can be written as

$$
f(\boldsymbol{x}) = \sum_{k = 1}^K\pi_kf(\boldsymbol{x}; \boldsymbol{\mu}_k, \Sigma_k)
$$

where each $f(\boldsymbol{x}; \boldsymbol{\mu}_k, \Sigma_k)$ is itself a
multivariate Gaussian distribution characterized by its mean,
$\boldsymbol{\mu}$ and covariance matrix, $\Sigma_k$. An often-used
simplifying assumption is that each component shares a common covariance
matrix, say $\Sigma_k = \Sigma$. The individual mixing proportions $\pi_k$
represent the probability of an individual observational unit belonging to the
$k^{th}$ cluster with $\sum_{k = 1}^K\pi_k = 1$, *i.e.* every unit belongs
to one of the clusters.

Although the theoretical details of Gaussian mixture models are well beyond
the scope of this text, the implementation details are relatively
straightforward. We will use the `GaussianMixture` class from `scikit-learn`
to fit a Gaussian mixture model using an Expectation-Maximization (EM)
algorithm and make cluster assignments. We will start with a two-component, or
two-cluster, mixture using the `age` and `xGxAp90` variables. We assume a
common covariance matrix in our initial example by setting
`covariance_type='tied'`. As before, we standardize the features prior to
fitting.

In [ ]:
nwsl_mix_df = nwsl_cluster_df.copy()

features_m = nwsl_mix_df[["age", "xGxAp90"]]
scaler_m = StandardScaler()
X_scaled_m = scaler_m.fit_transform(features_m)

gmm = GaussianMixture(n_components=2, covariance_type="tied",
                       random_state=109823)
gmm.fit(X_scaled_m)

The individual observations are clustered according to their estimated
posterior probabilities of cluster assignment. The `predict_proba` method
returns these estimates.

In [ ]:
posteriors = gmm.predict_proba(X_scaled_m)
pd.DataFrame(posteriors, columns=["Comp 1", "Comp 2"]).head(6)

The first line in the posterior output shows the probability of belonging to
each of the two components. These numbers represent the posterior probability
of belonging to the first and second components, or clusters. The model
classifies each player based on which cluster's posterior probability is
higher. This is referred to as a "soft" clustering given that we are assigning
group membership based on the posterior probabilities.

In [ ]:
nwsl_mix_df[["age", "xGxAp90"]].head(6)

We can examine the cluster centers by transforming the fitted means back to
the original scale:

In [ ]:
centers_scaled = gmm.means_
centers_original = scaler_m.inverse_transform(centers_scaled)
centers_df = pd.DataFrame(centers_original, columns=["age", "xGxAp90"],
                           index=["Comp 1", "Comp 2"])
centers_df

Let's add the centers to our scatterplot and color the points according to
their classification. The first variable (`pp`) would be our soft
classification probability whereas the `class` variable is a hard
classification. We can visualize both using some of the tools that we have
used previously. The centers, or means, of the individual cluster
distributions are given as the solid black triangles. @fig-soccer-mm shows the
points with shapes representing their cluster membership whereas
@fig-soccer-mm-2 shows the points colored by their probability of cluster
membership as a continuum.

In [ ]:
#| label: fig-soccer-mm
#| fig-cap: "Player age (x) versus expected goals plus assists per 90 (y) with point shapes representing cluster membership based on the two-component Gaussian mixture model."

nwsl_mix_df["pp"] = posteriors[:, 0]
nwsl_mix_df["class"] = np.where(nwsl_mix_df["pp"] <= 0.5, "one", "two")

fig, ax = plt.subplots(figsize=(8, 5))
class_markers = {"one": "o", "two": "s"}
palette = sns.color_palette("Set1", 2)
for i, cls in enumerate(["one", "two"]):
    subset = nwsl_mix_df[nwsl_mix_df["class"] == cls]
    ax.scatter(subset["age"], subset["xGxAp90"], marker=class_markers[cls],
               color=palette[i], alpha=0.7, label=f"Cluster {cls}")

ax.scatter(centers_original[:, 0], centers_original[:, 1],
           marker="^", color="black", s=100, zorder=5)
ax.set_xlabel("Age of Players")
ax.set_ylabel("Expected Goals plus Assists per 90")
ax.set_title("Gaussian Mixture Clustering of NWSL players")
ax.legend(title="Cluster")
plt.show()

In [ ]:
#| label: fig-soccer-mm-2
#| fig-cap: "Player age (x) versus expected goals plus assists per 90 (y) with point color representing probability of cluster membership based on the two-component Gaussian mixture model."

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(nwsl_mix_df["age"], nwsl_mix_df["xGxAp90"],
                c=nwsl_mix_df["pp"], cmap="Spectral", vmin=0, vmax=1,
                alpha=0.8)
ax.scatter(centers_original[:, 0], centers_original[:, 1],
           marker="^", color="black", s=100, zorder=5)
plt.colorbar(sc, ax=ax, label="Probability (Comp 1)")
ax.set_xlabel("Age")
ax.set_ylabel("Expected Goals and Assists per 90")
plt.show()

The `weights_` attribute in the `GaussianMixture` object yields the estimated
mixing proportions for each component.

In [ ]:
gmm.weights_

One thing to note about using `GaussianMixture` specifically or EM algorithms
in general is that you should always initialize the algorithm using multiple
initial values in order to ensure convergence to a global maximum. You can
check the log-likelihood in the output to see if you (hopefully) arrive at a
global maximum. Consider a different initialization of the same function call
and examine the log-likelihood.

In [ ]:
gmm_2 = GaussianMixture(n_components=2, covariance_type="tied",
                          random_state=42)
gmm_2.fit(X_scaled_m)

print(f"Model 1 log-likelihood: {gmm.score(X_scaled_m) * len(X_scaled_m):.2f}")
print(f"Model 2 log-likelihood: {gmm_2.score(X_scaled_m) * len(X_scaled_m):.2f}")

___

## Case Study: Soccer Roster Economics

Soccer teams throughout the world are no different than other teams we have
examined in that they are businesses that need to operate within a budget and
make a financial return for their owners and/or shareholders. Some teams have
very wealthy owners, especially in England where in the 21st century club
ownership has become a status symbol for royalty, oligarchs, and sovereign
wealth funds. Anyone can commission a superyacht, but there are only 20 teams
in the English Premier League.

While teams in the biggest European leagues do not have the same salary cap
restrictions of NBA, NFL, or NHL teams, they are subject to UEFA Financial
Fair Play (FFP) restrictions. These rules were created in 2010 to ensure that
clubs did not spend so much money that threatened their existence as going
concerns. FFP was implemented after a 2009 study showed that approximately
half of European clubs were losing money, and as many as 20% were in financial
danger. FFP rules have continued to evolve over the years, but currently state
that clubs can only lose a total of 60 million euros over three years. They
also have somewhat of a spending cap (as opposed to a salary cap), where
salaries, transfer fees (fees paid to another club when acquiring playing
rights to that player), and money paid to player agents can not exceed 70% of
the club's total revenue. Of course these rules are frequently broken, perhaps
most often by the richest clubs, but FFP rule enforcement is not the purpose of
this section.

The purpose of this section is to talk about the need for the administration of
soccer teams to think carefully about the players that they acquire and the
salaries that they pay in order to make their team competitive both on the
pitch and in the board room. While the rules these teams play under may seem
unfair, it also should be familiar by now. The subtitle of Michael Lewis'
groundbreaking sports analytics book Moneyball is "The Art of Winning an Unfair
Game." While the rules are not directly different for different teams in
American baseball as they are in European soccer, the same techniques for
identifying value in players are even more important. Billy Beane's thinking
in Moneyball was that general managers should be buying runs, and not players.
Perhaps the same thinking should be applied to buying goals, or buying xG?

So, if you readers of the text are administration and general managers of
soccer clubs, how do you go about accurately valuing the fair worth and
appropriate salary for a player? One approach is to use player valuations from
[Transfermarkt.com](https://www.transfermarkt.com/). Join these valuations with
the `epl_player_stats_2022` data from the `isapy_data` module and perform a
full accounting of valuations in order to identify under- and over-paid
players.

In [ ]:
#| eval: false

# Example: scraping EPL valuations from Transfermarkt
# Note: Web scraping Transfermarkt requires careful handling of their
# Terms of Service and rate limiting. The code below is illustrative.

import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}
url = "https://www.transfermarkt.com/premier-league/marktwerte/wettbewerb/GB1"
response = requests.get(url, headers=headers)

In [ ]:
#| eval: false

epl_player_stats_2022 = isapy_data.load_data("epl_player_stats_2022")
# epl_stats_val = epl_player_stats_2022.merge(
#     epl_valuations, left_on="player", right_on="player_name", how="left"
# )

Choose the covariates that you think would be appropriate. Some might include
`age`, `position`, `xG`, `xA`, among other characteristics. Do some
calculations to determine which players you believe are overvalued and
undervalued. What else do you notice? What strategies could you derive from
this analysis?

___

## Lab: Goalkeeper Styles in the EPL

In this lab, we will investigate playing styles by goalkeepers in the English
Premier League. The data for the lab can be found in the `isapy_data` module
under `epl_gk_stats_2022`. The data set consists of 42 observations
(goalkeepers) across 32 variables. We will perform a k-means clustering of
the players to gain insights into their shot stopping and passing styles.

1. Create a k-means clustering (two clusters) of goalkeepers based on the
following variables: post shot expected goals per shot on target
(`post_shot_xG_per_shot_on_target`), launch completion percent
(`launch_completion_percent`), pass launch percentage (`pass_launch_percent`),
average pass length (`avg_pass_len`), goal kick launch percent
(`gk_launch_percent`), and defensive actions outside the penalty area per 90
minutes (`def_actions_out_pen_area_p90`). Standardize the features before
clustering. Set your seed at 8675.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_gk_stats_2022 = isapy_data.load_data("epl_gk_stats_2022")

gk_features = epl_gk_stats_2022[[
    "post_shot_xG_per_shot_on_target", "launch_completion_percent",
    "pass_launch_percent", "avg_pass_len", "gk_launch_percent",
    "def_actions_out_pen_area_p90"
]].dropna()

scaler_gk = StandardScaler()
X_scaled_gk = scaler_gk.fit_transform(gk_features)

km_gk = KMeans(n_clusters=2, random_state=8675, n_init=10)
gk_clusters = km_gk.fit_predict(X_scaled_gk)
print(f"Cluster sizes: {pd.Series(gk_clusters).value_counts().to_dict()}")

2. Create a data frame showing the players, their nationality, and their
cluster assignment. Sort the data by cluster.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_gk_cluster_df = epl_gk_stats_2022.loc[gk_features.index].copy()
epl_gk_cluster_df["k_means"] = gk_clusters

(epl_gk_cluster_df[["player", "nation", "k_means"]]
 .sort_values("k_means"))

3. Plot the average pass length by the number of defensive actions outside
the penalty area per ninety minutes. Color the points by the cluster
membership.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette("Set1", 2)
for i, cluster in enumerate(sorted(epl_gk_cluster_df["k_means"].unique())):
    subset = epl_gk_cluster_df[epl_gk_cluster_df["k_means"] == cluster]
    ax.scatter(subset["avg_pass_len"], subset["def_actions_out_pen_area_p90"],
               color=palette[i], label=f"Cluster {cluster + 1}")
ax.set_xlabel("Average Pass Length")
ax.set_ylabel("Defensive Actions Outside the Penalty Area per 90")
ax.set_title("Clustering Goalkeepers in the EPL")
ax.legend(title="Cluster")
plt.show()

4. Add simple linear regression trend lines for each cluster. What can you
infer based on these trend lines?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette("Set1", 2)
for i, cluster in enumerate(sorted(epl_gk_cluster_df["k_means"].unique())):
    subset = epl_gk_cluster_df[epl_gk_cluster_df["k_means"] == cluster]
    ax.scatter(subset["avg_pass_len"], subset["def_actions_out_pen_area_p90"],
               color=palette[i], label=f"Cluster {cluster + 1}")
    # Add trend line
    x = subset["avg_pass_len"].values.reshape(-1, 1)
    y = subset["def_actions_out_pen_area_p90"].values
    reg = LinearRegression().fit(x, y)
    x_line = np.linspace(x.min(), x.max(), 100).reshape(-1, 1)
    ax.plot(x_line, reg.predict(x_line), color=palette[i], linewidth=2)
ax.set_xlabel("Average Pass Length")
ax.set_ylabel("Defensive Actions Outside the Penalty Area per 90")
ax.set_title("Clustering Goalkeepers in the EPL")
ax.legend(title="Cluster")
plt.show()

## Exercises

Problems 1 -- 5 of the exercises will use the `epl_team_stats` data set in
the `isapy_data` module. Exercise 6 will use `epl_player_stats_2022`.

1. Plot average age of a team (`age`) versus expected goals and assists per 90
minutes (`xG_xA_p90`).

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_team_stats = isapy_data.load_data("epl_team_stats")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(epl_team_stats["age"], epl_team_stats["xG_xA_p90"], alpha=0.5)
ax.set_xlabel("Average Age of Players on Team")
ax.set_ylabel("Expected Goals and Assists per 90 Minutes")
ax.set_title("Offensive Production vs Average Team Age in the EPL")
plt.show()

2. Add a smooth trend line to the plot given in (1) and comment on your
observations.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(epl_team_stats["age"], epl_team_stats["xG_xA_p90"], alpha=0.5)
sns.regplot(data=epl_team_stats, x="age", y="xG_xA_p90",
            lowess=True, scatter=False, color="black", ax=ax)
ax.set_xlabel("Average Age of Players on Team")
ax.set_ylabel("Expected Goals and Assists per 90 Minutes")
ax.set_title("Offensive Production vs Average Team Age in the EPL")
plt.show()

3. Label all points that are above 3.5 in `xG_xA_p90` by team and season.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_team_stats["team_season"] = np.where(
    epl_team_stats["xG_xA_p90"] > 3.5,
    epl_team_stats["squad"] + " " + epl_team_stats["season"],
    ""
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(epl_team_stats["age"], epl_team_stats["xG_xA_p90"], alpha=0.5)
sns.regplot(data=epl_team_stats, x="age", y="xG_xA_p90",
            lowess=True, scatter=False, color="black", ax=ax)
labeled = epl_team_stats[epl_team_stats["team_season"] != ""]
for _, row in labeled.iterrows():
    ax.annotate(row["team_season"], (row["age"], row["xG_xA_p90"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("Average Age of Players on Team")
ax.set_ylabel("Expected Goals and Assists per 90 Minutes")
ax.set_title("Offensive Production vs Average Team Age in the EPL")
plt.show()

4. Remove Manchester City and redo the trend plot. Add a linear trend line to
the plot as well.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_no_mc = epl_team_stats[epl_team_stats["squad"] != "Manchester City"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(epl_no_mc["age"], epl_no_mc["xG_xA_p90"], alpha=0.5)
sns.regplot(data=epl_no_mc, x="age", y="xG_xA_p90",
            lowess=True, scatter=False, color="blue", ax=ax)
sns.regplot(data=epl_no_mc, x="age", y="xG_xA_p90",
            scatter=False, color="black", ci=95, ax=ax)
ax.set_xlabel("Average Age of Players on Team")
ax.set_ylabel("Expected Goals and Assists per 90 Minutes")
ax.set_title("Offensive Production vs Average Team Age in the EPL (minus Manchester City)")
plt.show()

5. Repeat problems (1) -- (2) using only data from 2018-19 through 2021-22.
Show each season separately as its own plot in a faceted graph.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_recent = epl_team_stats[epl_team_stats["season"] != "2017-18"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, season in enumerate(sorted(epl_recent["season"].unique())):
    ax = axes[i]
    subset = epl_recent[epl_recent["season"] == season]
    ax.scatter(subset["age"], subset["xG_xA_p90"], alpha=0.5)
    sns.regplot(data=subset, x="age", y="xG_xA_p90",
                lowess=True, scatter=False, color="black", ax=ax)
    ax.set_xlabel("Average Age of Players on Team")
    ax.set_ylabel("xG + xA per 90")
    ax.set_title(season)
plt.tight_layout()
plt.show()

6. Using the `epl_player_stats_2022` data frame from the `isapy_data` module,
plot expected goals plus assists per 90 (`xG_xA_p90`) on the y axis and
minutes played (`min`) on the x axis. Label the players having an `xG_xA_p90`
value greater than one.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_player_stats_2022 = isapy_data.load_data("epl_player_stats_2022")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(epl_player_stats_2022["min"],
           epl_player_stats_2022["xG_xA_p90"], alpha=0.5)
sns.regplot(data=epl_player_stats_2022, x="min", y="xG_xA_p90",
            scatter=False, color="blue", ax=ax)
labeled = epl_player_stats_2022[epl_player_stats_2022["xG_xA_p90"] > 1]
for _, row in labeled.iterrows():
    ax.annotate(row["player"], (row["min"], row["xG_xA_p90"]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.set_xlabel("Total Minutes Played 2021-2022")
ax.set_ylabel("Expected Goals and Assists per 90 Minutes")
ax.set_title("xG + xA p90 in the EPL 2021-2022")
plt.show()

7. Filter the data set to players who played more than 90 minutes. Regress
expected goals plus assists per 90 (`xG_xA_p90`) on minutes played (`min`).
Does the result surprise you?

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

import statsmodels.api as sm

epl_filtered = epl_player_stats_2022[epl_player_stats_2022["min"] > 90].copy()
X = sm.add_constant(epl_filtered["min"])
y = epl_filtered["xG_xA_p90"]
model = sm.OLS(y, X).fit()
print(model.summary())

The insignificant slope parameter (minutes) should not be a surprise given
that the dependent variable has been normalized based on a per 90 minute
criterion.

8. Redo problems (6) and (7) using expected goals plus expected assists as the
dependent variable.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

epl_filtered["xG_xA"] = epl_filtered["xG"] + epl_filtered["xA"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(epl_filtered["min"], epl_filtered["xG_xA"], alpha=0.5)
sns.regplot(data=epl_filtered, x="min", y="xG_xA",
            scatter=False, color="blue", ax=ax)
ax.set_xlabel("Total Minutes Played 2021-2022")
ax.set_ylabel("Expected Goals + Assists")
plt.show()

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| eval: false

X = sm.add_constant(epl_filtered["min"])
y = epl_filtered["xG_xA"]
model_2 = sm.OLS(y, X).fit()
print(model_2.summary())

9. Given the previous exercises, what are your overall interpretations?